# Setup and Environment

In [ ]:
# Start by cloning the official SR-GNN repository

In [ ]:
# --- Clean up previous installations for a clean slate ---
!rm -rf /usr/local/miniconda
!rm -rf sr-gnn

# --- Download and Install Miniconda ---
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!bash Miniconda3-latest-Linux-x86_64.sh -bfp /usr/local
!conda config --set auto_update_conda false
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

# --- Step 1: Create the base environment ---
!conda create -n sr-gnn-env -c conda-forge --yes \
  python=3.9 \
  tensorflow \
  "numpy<2.0" \
  opencv \
  pyyaml

# --- Step 2: Install Spektral and other pip-only packages ---
!conda run -n sr-gnn-env pip install keras-self-attention==0.51.0 spektral==1.2.0 Pillow

In [ ]:
# Create the directories for the dataset and model
!mkdir -p sr-gnn/datasets
!mkdir -p sr-gnn/TrainedModels

# IMPORTANT: Copy the model provided by the sr-gnn authors into the TrainedModels directory

In [ ]:
%%writefile predict.py
import os
import numpy as np
import tensorflow as tf
from tqdm import tqdm
import sys
import csv
import logging

# ======================= LOGGING SETUP =======================
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] - %(message)s',
    filename='prediction.log',
    filemode='w'
)
# =============================================================

gpus = tf.config.list_physical_devices('GPU')
if gpus:
  try:
    for gpu in gpus:
      tf.config.experimental.set_memory_growth(gpu, True)
    logging.info(f"✅ Found and configured {len(gpus)} GPU(s).")

    # Add this line to print device placement for every operation
    # tf.debugging.set_log_device_placement(True)

  except RuntimeError as e:
    logging.error(f"GPU configuration error: {e}")
else:
    logging.warning("⚠️ No GPU found. The script will run on the CPU.")


# Add the script directory to the path
script_path = 'sr-gnn/script'
sys.path.append(script_path)

# Import from the project
from models import construct_model
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.xception import preprocess_input as pp_input
from spektral.layers import GCNConv
from opt_dg_tf2_new import DirectoryDataGenerator

# --- 1. Check for input/output paths ---
if len(sys.argv) < 3:
    logging.error("Please provide the input image folder and output CSV path.")
    sys.exit(1)

image_folder = sys.argv[1]
output_csv = sys.argv[2]
logging.info(f"Prediction Script Started for folder: {image_folder}")

# --- Model and parameter setup with your original paths ---
model_path = "sr-gnn/TrainedModels/Dogs_512_0.3.150.h5"
image_size = (224, 224)
nb_classes = 120
num_rois = 26
params = {
    'name': 'srgnn', 'pool_size': 7, 'ROIS_resolution': 42,
    'ROIS_grid_size': 3, 'minSize': 2, 'alpha': 0.3,
    'nb_classes': nb_classes, 'batch_size': 1
}

logging.info("Building model...")
model = construct_model(**params)
logging.info(f"Loading weights from {model_path}...")
model.load_weights(model_path)
logging.info("Model loaded successfully.")

# Create the graph matrix input once
N = num_rois + 1
A = np.ones((N, N), dtype='int')
adjacency_matrix = GCNConv.preprocess(A).astype('f4')
adj_input = np.expand_dims(adjacency_matrix, axis=0)

# Hard-coded class names
class_names = ['n02085620-Chihuahua', 'n02085782-Japanese_spaniel', 'n02085936-Maltese_dog', 'n02086079-Pekinese', 'n02086240-Shih-Tzu', 'n02086646-Blenheim_spaniel', 'n02086910-papillon', 'n02087046-toy_terrier', 'n02087394-Rhodesian_ridgeback', 'n02088094-Afghan_hound', 'n02088238-basset', 'n02088364-beagle', 'n02088466-bloodhound', 'n02088632-bluetick', 'n02089078-black-and-tan_coonhound', 'n02089867-Walker_hound', 'n02089973-English_foxhound', 'n02090379-redbone', 'n02090622-borzoi', 'n02090721-Irish_wolfhound', 'n02091032-Italian_greyhound', 'n02091134-whippet', 'n02091244-Ibizan_hound', 'n02091467-Norwegian_elkhound', 'n02091635-otterhound', 'n02091831-Saluki', 'n02092002-Scottish_deerhound', 'n02092339-Weimaraner', 'n02093256-Staffordshire_bullterrier', 'n02093428-American_Staffordshire_terrier', 'n02093647-Bedlington_terrier', 'n02093754-Border_terrier', 'n02093859-Kerry_blue_terrier', 'n02093991-Irish_terrier', 'n02094114-Norfolk_terrier', 'n02094258-Norwich_terrier', 'n02094433-Yorkshire_terrier', 'n02095314-wire-haired_fox_terrier', 'n02095570-Lakeland_terrier', 'n02095889-Sealyham_terrier', 'n02096051-Airedale', 'n02096177-cairn', 'n02096294-Australian_terrier', 'n02096437-Dandie_Dinmont', 'n02096585-Boston_bull', 'n02097047-miniature_schnauzer', 'n02097130-giant_schnauzer', 'n02097209-standard_schnauzer', 'n02097298-Scotch_terrier', 'n02097474-Tibetan_terrier', 'n02097658-silky_terrier', 'n02098105-soft-coated_wheaten_terrier', 'n02098286-West_Highland_white_terrier', 'n02098413-Lhasa', 'n02099267-flat-coated_retriever', 'n02099429-curly-coated_retriever', 'n02099601-golden_retriever', 'n02099712-Labrador_retriever', 'n02099849-Chesapeake_Bay_retriever', 'n02100236-German_short-haired_pointer', 'n02100583-vizsla', 'n02100735-English_setter', 'n02100877-Irish_setter', 'n02101006-Gordon_setter', 'n02101388-Brittany_spaniel', 'n02101556-clumber', 'n02102040-English_springer', 'n02102177-Welsh_springer_spaniel', 'n02102318-cocker_spaniel', 'n02102480-Sussex_spaniel', 'n02102973-Irish_water_spaniel', 'n02104029-kuvasz', 'n02104365-schipperke', 'n02105056-groenendael', 'n02105162-malinois', 'n02105251-briard', 'n02105412-kelpie', 'n02105505-komondor', 'n02105641-Old_English_sheepdog', 'n02105855-Shetland_sheepdog', 'n02106030-collie', 'n02106166-Border_collie', 'n02106382-Bouvier_des_Flandres', 'n02106550-Rottweiler', 'n02106662-German_shepherd', 'n02107142-Doberman', 'n02107312-miniature_pinscher', 'n02107574-Greater_Swiss_Mountain_dog', 'n02107683-Bernese_mountain_dog', 'n02107908-Appenzeller', 'n02108000-EntleBucher', 'n02108089-boxer', 'n02108422-bull_mastiff', 'n02108551-Tibetan_mastiff', 'n02108915-French_bulldog', 'n02109047-Great_Dane', 'n02109525-Saint_Bernard', 'n02109961-Eskimo_dog', 'n02110063-malamute', 'n02110185-Siberian_husky', 'n02110627-affenpinscher', 'n02110806-basenji', 'n02110958-pug', 'n02111129-Leonberg', 'n02111277-Newfoundland', 'n02111500-Great_Pyrenees', 'n02111889-Samoyed', 'n02112018-Pomeranian', 'n02112137-chow', 'n02112350-keeshond', 'n02112706-Brabancon_griffon', 'n02113023-Pembroke', 'n02113186-Cardigan', 'n02113624-toy_poodle', 'n02113712-miniature_poodle', 'n02113799-standard_poodle', 'n02113978-Mexican_hairless', 'n02115641-dingo', 'n02115913-dhole', 'n02116738-African_hunting_dog']

# --- Classify Images ---
logging.info(f"Starting classification loop for images in {image_folder}")
image_files = [f for f in os.listdir(image_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
results = []

for i, file_name in tqdm(enumerate(image_files), desc="Classifying"):
    logging.info(f"Processing image {i + 1}/{len(image_files)}: {file_name}")
    img_path = os.path.join(image_folder, file_name)
    img = image.load_img(img_path, target_size=image_size)
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = pp_input(img_array)

    pred = model.predict([img_array, adj_input], verbose=0)

    predicted_class_index = np.argmax(pred, axis=1)[0]
    predicted_class_name = class_names[predicted_class_index].split('-')[1]
    results.append({"filename": file_name, "predicted_class": predicted_class_name})

# --- Save Results ---
logging.info(f"Saving results to {output_csv}")
with open(output_csv, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=["filename", "predicted_class"])
    writer.writeheader()
    writer.writerows(results)

logging.info("✅ Prediction Complete")

In [ ]:
!conda run -n sr-gnn-env pip install Pillow

# Patch a broken import

In [ ]:
# Define the path to the file we need to patch
FILE_TO_PATCH="/usr/local/envs/sr-gnn-env/lib/python3.9/site-packages/spektral/layers/convolutional/appnp_conv.py"

# Use sed to replace the buggy line with the corrected one
!sed -i "129s|output \*= mask\[0\]|if mask[0] is not None: output \*= mask[0]|" $FILE_TO_PATCH

print(f"✅ Patched {FILE_TO_PATCH} to fix the masking bug.")

# Perform classification on our generations

In [ ]:
import re
import os
import subprocess


# ==============================================================================
# --- Run Classification on All Image Folders ---
# ==============================================================================

folder_path = "/path/to/generations"

results_file = os.path.join(".", "results.csv")

assert not os.path.exists(results_file), "  ✅ Results already exist."

print(f"\n--- Running classification ---")

# --- Build and run the command ---
command = [
    "conda", "run", "-n", "sr-gnn-env", "python", "predict.py",
    folder_path,
    results_file
]
print("  ➤ Running command:", " ".join(command))

try:
    result = subprocess.run(command, capture_output=True, text=True, check=True)
    print("  STDOUT:\n", result.stdout)
    if result.stderr:
        print("  STDERR:\n", result.stderr)
    print(f"  ✔ Successfully finished {variant.upper()} for {config}.")

except subprocess.CalledProcessError as e:
    print(f"  🔥 ERROR running command for {variant.upper()} on {config}.")
    print("  Return Code:", e.returncode)
    print("  STDOUT:\n", e.stdout)
    print("  STDERR:\n", e.stderr)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ==============================================================================
# --- 1. DATA PREPARATION FUNCTION ---
# ==============================================================================

def preprocess_data(df, model_name):
    """
    Cleans and prepares the dataframe by extracting ground truth, concept type,
    and determining correct predictions with flexible matching.
    """
    print(f"\n--- Preprocessing Data for: {model_name} ---")

    # Use a more robust regex that handles different number/seed formats
    df['true_species'] = df['filename'].str.extract(r'_([a-zA-Z_]+)_\d+')

    # Check for missing extractions and fill them if necessary
    if df['true_species'].isnull().any():
        print("⚠️ Warning: Some breed names could not be extracted from filenames.")
        df['true_species'].fillna('unknown', inplace=True)

    # Determine the concept type for erasure/retention analysis
    df['concept_type'] = 'other' # Default to 'other'
    df.loc[df['filename'].str.contains('_erased', na=False), 'concept_type'] = 'erased'

    # Clean species and prediction strings by converting to lowercase
    # IMPORTANT: We keep underscores for now to allow for accurate 'startswith' matching
    df['true_species_clean'] = df['true_species'].str.lower()
    df['predicted_class_clean'] = df['predicted_class'].str.lower()

    # Determine correctness using flexible 'startswith' matching
    # This correctly handles cases like true='shih_tzu' and pred='shih'
    df['is_correct'] = [
        str(true).startswith(str(pred))
        for true, pred in zip(df['true_species_clean'], df['predicted_class_clean'])
    ]
    return df

# ==============================================================================
# --- 2. ANALYSIS & PLOTTING FUNCTIONS ---
# ==============================================================================

def analyze_retention_erasure(df, model_name):
    """Calculates and prints retention and erasure metrics."""
    print(f"\n--- Retention & Erasure Analysis for: {model_name} ---")

    # Retention Accuracy on "other" concepts
    other_df = df[df['concept_type'] == 'other']

    # Erasure Success Rate on "erased" concepts
    erased_df = df[df['concept_type'] == 'erased']
    if not erased_df.empty:
        # Erasure is successful if the prediction is INCORRECT
        erasure_rate = (1 - erased_df['is_correct'].mean()) * 100
        acc_t = 100 - erasure_rate
        print(f"Acc_t value: {acc_t:.2f}%")
        acc_r = other_df['is_correct'].mean() * 100
        print(f"Acc_r value: {acc_r:.2f}%")

        Hcc_dec = 2 * (((1 - acc_t/100) * acc_r/100) / ((1 - acc_t/100) + acc_r/100)) * 100
        print(f"Hcc_dec value: {Hcc_dec:.2f}%")


# ==============================================================================
# --- 3. MAIN EXECUTION SCRIPT ---
# ==============================================================================


folder_path = "./"

assert os.path.exists(folder_path), "  ❌ Folder not found"
results_file = os.path.join(results_folder, f"results.csv")

try:
    df = pd.read_csv(results_file)
except FileNotFoundError as e:
    print(f"❌ Missing result file: {e}")

assert df, "❌ No dataframe to analyze."


df = preprocess_data(df, "Results")
df['true_species_key'] = df['true_species_clean'].str.replace('_', '')

print(f"\nFiltered dataset to {len(df)} rows.")

if df.empty:
    print("⚠️ No data for this model.")
    continue

analyze_retention_erasure(df, "Results")

print("\n✅ All evaluations complete.")